# Music-seeder remote CPU validation

Run this notebook on Google Colab or another hosted Jupyter CPU runtime. Python is only the notebook control plane; every project build and test command is C++23. No GPU is requested.

Set `REF` to a branch or commit. A commit hash gives reproducible evidence. The toolchain cell installs GCC 14 because Colab's default libstdc++ may not provide C++23 `<expected>`.

In [ ]:
import os
os.environ["REPOSITORY"] = "https://github.com/Idle0dreamer/Music-seeder.git"
os.environ["REF"] = "main"
os.environ["WORKERS"] = "2"
os.environ["CXX"] = "g++-14"

In [ ]:
%%bash
set -euo pipefail
sudo apt-get update -qq
sudo apt-get install -y -qq software-properties-common
sudo add-apt-repository -y ppa:ubuntu-toolchain-r/test
sudo apt-get update -qq
sudo apt-get install -y -qq g++-14
"$CXX" --version
printf '#include <expected>\nint main(){std::expected<int, int> value{1}; return *value - 1;}\n' |
  "$CXX" -std=c++23 -Wall -Wextra -Wpedantic -Werror -x c++ -fsyntax-only -

In [ ]:
%%bash
set -euo pipefail
rm -rf /content/Music-seeder
git clone --filter=blob:none "$REPOSITORY" /content/Music-seeder
git -C /content/Music-seeder checkout "$REF"
git -C /content/Music-seeder rev-parse HEAD

The next cell performs the debug law suite, optimized build, undefined-behavior sanitizer, and representative seeded CLI runs. It stops on the first failure.

In [ ]:
%%bash
set -euo pipefail
cd /content/Music-seeder
make -s -j"$WORKERS" CXX="$CXX" kernel-test
make -s -j"$WORKERS" CXX="$CXX" kernel
make -s -j"$WORKERS" CXX="$CXX" kernel-sanitize
for seed in 0 1 2 3 4; do
  ./build/kernel "$seed"
done

If all cells pass, record the printed commit hash and results in `STATUS.md`. Do not describe a different commit as validated.